# HyenaDNA test

In [1]:
import pandas as pd
import os

## Load data

In [2]:
base_path = "../../data/perphect-data"

couples_df = pd.read_csv(os.path.join(base_path, "public_data_set", "couples_df.csv"))
phages_df = pd.read_csv(os.path.join(base_path, "public_data_set", "phages_df.csv"))
bacteria_df = pd.read_csv(os.path.join(base_path, "public_data_set", "bacteria_df.csv"))
print('couples_df.shape = ', couples_df.shape)
print('phages_df.shape = ', phages_df.shape)
print('bacteria_df.shape = ', bacteria_df.shape)

couples_df.shape =  (4202, 4)
phages_df.shape =  (3459, 3)
bacteria_df.shape =  (94, 3)


In [3]:
phages_df.head()

,phage_id,phage_sequence,sequence_length
0,5326,TGCGGCCGCCCCATCCTGTACGGGTTTCCAAGTCGATCGGAGGGCA...,53124
1,6247,GGCTTTCGTGTGAGCCGTGATGTTTTCACGAATATGTGCCCCACCT...,74483
2,1976,GTGGGAATTTTTTTTTTGGGTTGCGCGGTGATCGCCGATGACGACG...,50781
3,430,ATGGCTTCGACTCAGACTCCAGCCGTCGGCAAGACCACGGCCATCG...,71565
4,431,TGCGGCTGAGCCATCGTGTACGGGTTTCCAAGTCCATCAGAGCCGG...,53396


## Embeddings using 🤗 transformers

In [1]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

In [2]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
device

'cuda:0'

In [3]:
from huggingface import HyenaDNAPreTrainedModel, CharacterTokenizer

# instantiate pretrained model
pretrained_model_name = 'hyenadna-medium-450k-seqlen'
max_length = 450_000

model = HyenaDNAPreTrainedModel.from_pretrained(
    './checkpoints',
    pretrained_model_name,
)

# create tokenizer, no training involved :)
tokenizer = CharacterTokenizer(
    characters=['A', 'C', 'G', 'T', 'N'],  # add DNA characters
    model_max_length=max_length,
)

/home/pere.carrillo/.local/share/mamba/envs/pbi-hyenadna/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")


Using device: cuda
Updated git hooks.
Git LFS initialized.


Cloning into 'hyenadna-small-32k-seqlen'...


Loaded pretrained weights ok!


NotImplementedError: 

In [ ]:
# create a sample
sequence = 'ACTG' * int(max_length/4)
tok_seq = tokenizer(sequence)["input_ids"]

2048

In [ ]:
# place on device, convert to tensor
tok_seq = torch.LongTensor(tok_seq).unsqueeze(0).to(device)  # unsqueeze for batch dim

2, 53124, 74483


torch.Size([2, 2048])

In [ ]:
# prep model and forward
model.to(device)
model.eval()  # deterministic

with torch.inference_mode():
    embeddings = model(tok_seq)

print(embeddings.shape)  # embeddings here!

torch.Size([2, 2048])

In [52]:
# Compute sequences embeddings
embeddings = torch_outs['hidden_states'][-1].detach().cpu().numpy()
print(f"Embeddings shape: {embeddings.shape}")
# print(f"Embeddings per token: {embeddings}")

# Add embed dimension axis
attention_mask_unsq = torch.unsqueeze(attention_mask, dim=-1).cpu()

# Compute mean embeddings per sequence
mean_sequence_embeddings = torch.sum(attention_mask_unsq*embeddings, axis=-2)/torch.sum(attention_mask_unsq, axis=1)
print(f"Mean sequence embeddings: {mean_sequence_embeddings}")

Embeddings shape: (2, 2048, 512)
Mean sequence embeddings: tensor([[-0.2326,  0.3941,  0.3469,  ..., -0.1369, -0.1470, -0.2098],
        [-0.2374,  0.4609,  0.3259,  ..., -0.2885, -0.1596, -0.1963]])
